# HACKOWEEK SEM 5 - Week 3 & Week 4
## Data Science Foundations: Python Essentials, NumPy, Pandas & Data Visualization

**Student:** Samruddhi Kalbande  
**Course:** B.Tech Computer Science / Information Technology (5th Semester)  
**Topics Covered:**
1. **Python Essentials:** Functions, Object-Oriented Programming (OOP), and List/Dict Comprehensions
2. **NumPy:** Multidimensional Arrays, Vectorized Operations, and Array Broadcasting
3. **Pandas:** DataFrames, Data Cleaning, Merging Datasets, and GroupBy Aggregations
4. **Data Visualization:** Exploratory plots using Matplotlib and Seaborn

**Dataset:** Curated Kaggle Student Performance & Academic Insights Dataset (`kaggle_student_performance.csv` and `course_enrollments.csv`)

### 1. Environment Setup & Data Loading
Let's import the necessary libraries and load the Kaggle dataset.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 5)

# Load datasets
df_students = pd.read_csv('../data/kaggle_student_performance.csv')
df_enrollments = pd.read_csv('../data/course_enrollments.csv')

print(f"Loaded {len(df_students)} student records and {len(df_enrollments)} course enrollments.")
df_students.head(3)

---
## 2. Python Essentials: Functions, OOP & Comprehensions
Demonstrating class design, encapsulation, and idiomatic Python comprehensions.

In [2]:
class StudentRecord:
    """
    Represents an undergraduate engineering student with academic evaluation methods.
    """
    def __init__(self, student_id, name, department, year, cgpa, attendance, skills):
        self.student_id = student_id
        self.name = name
        self.department = department
        self.year = int(year)
        self._cgpa = float(cgpa)  # Encapsulated attribute
        self.attendance = float(attendance)
        self.skills = skills.split(';') if isinstance(skills, str) else skills

    @property
    def cgpa(self):
        return self._cgpa

    @property
    def academic_standing(self):
        if self._cgpa >= 9.0:
            return "Distinction"
        elif self._cgpa >= 8.0:
            return "First Class"
        elif self._cgpa >= 7.0:
            return "Second Class"
        return "Pass"

    def is_eligible_for_placement(self, min_cgpa=8.0, min_attendance=85.0):
        return self._cgpa >= min_cgpa and self.attendance >= min_attendance

    def __repr__(self):
        return f"<Student {self.student_id}: {self.name} | CGPA: {self._cgpa} | {self.academic_standing}>"

# 1. List Comprehension: Transform DataFrame rows into OOP instances
student_objects = [
    StudentRecord(
        row['StudentID'], row['Name'], row['Department'], row['Year'],
        row['CGPA'], row['AttendanceRate'], row['Skills']
    )
    for _, row in df_students.iterrows()
]

# 2. Dict Comprehension: Map StudentID -> Placement Eligibility
eligibility_dict = {s.student_id: s.is_eligible_for_placement() for s in student_objects}

# 3. List comprehension filter for distinction students
distinction_students = [s for s in student_objects if s.academic_standing == "Distinction"]

print(f"Total instantiated OOP records: {len(student_objects)}")
print(f"Distinction achievers ({len(distinction_students)}):", [s.name for s in distinction_students])
print(f"Placement eligible count: {sum(eligibility_dict.values())}/{len(student_objects)}")

---
## 3. NumPy: Arrays, Broadcasting & Vectorized Operations
NumPy offers C-speed vectorized numerical routines without slow Python for-loops.

In [3]:
# Extract standardized exam scores into a 2D NumPy array
scores_matrix = df_students[['MathScore', 'ReadingScore', 'WritingScore']].to_numpy()
print("Scores Matrix Shape (N, D):", scores_matrix.shape)

# Vectorized Row-wise Mean: Composite Exam Score per student
composite_scores = np.mean(scores_matrix, axis=1)

# Vectorized Column-wise Stats: Mean and Std per subject
subject_means = np.mean(scores_matrix, axis=0)
subject_stds = np.std(scores_matrix, axis=0)

print("Subject Means [Math, Reading, Writing]:", np.round(subject_means, 2))
print("Subject Stds  [Math, Reading, Writing]:", np.round(subject_stds, 2))

# --- ARRAY BROADCASTING ---
# Standardizing exam scores via Z-score: (X - mu) / sigma
# Shape: (20, 3) broadcasted against (3,)
z_scores = (scores_matrix - subject_means) / subject_stds
print("\nSample Standardized Z-Scores (First 3 students):\n", np.round(z_scores[:3], 2))

# Boolean Masking: Students with composite score >= 90.0 AND CGPA >= 9.0
cgpa_arr = df_students['CGPA'].to_numpy()
top_tier_mask = (composite_scores >= 90.0) & (cgpa_arr >= 9.0)
top_students = df_students.loc[top_tier_mask, ['StudentID', 'Name', 'Department', 'CGPA']]
print("\nTop-Tier Students Filtered via NumPy Boolean Mask:")
top_students

---
## 4. Pandas: DataFrames, Cleaning, Merging & GroupBy
Pandas enables end-to-end data manipulation and relational operations.

In [4]:
# 1. Data Cleaning & Inspection
print("Missing values check:")
print(df_students.isnull().sum())

# 2. Merging (Inner Join): Join Students with their Course Enrollments
df_merged = pd.merge(df_students, df_enrollments, on='StudentID', how='inner')
print(f"\nMerged dataframe shape: {df_merged.shape}")

# 3. GroupBy Aggregation: Department-level Academic Benchmarking
dept_stats = df_students.groupby('Department').agg(
    Total_Students=('StudentID', 'count'),
    Mean_CGPA=('CGPA', 'mean'),
    Max_CGPA=('CGPA', 'max'),
    Mean_Attendance=('AttendanceRate', 'mean'),
    Mean_Study_Hours=('StudyHoursPerWeek', 'mean')
).round(2)

print("\nDepartment Aggregations:")
dept_stats

In [5]:
# Course-level performance aggregation from merged dataset
course_stats = df_merged.groupby(['CourseCode', 'CourseName']).agg(
    Total_Enrollments=('StudentID', 'count'),
    Average_Grade_Points=('GradePoints', 'mean'),
    Credits=('Credits', 'first')
).round(2).reset_index()

course_stats.sort_values(by='Average_Grade_Points', ascending=False)

---
## 5. Data Visualization: Matplotlib & Seaborn
Creating informative, publication-quality visualizations for academic insights.

In [6]:
# Plot 1: CGPA Distribution with KDE curve
plt.figure(figsize=(9, 4.5))
sns.histplot(df_students['CGPA'], kde=True, color='royalblue', bins=8)
plt.axvline(df_students['CGPA'].mean(), color='crimson', linestyle='--', label=f'Mean CGPA ({df_students["CGPA"].mean():.2f})')
plt.title('Distribution of Student CGPA with KDE Curve', fontsize=12, fontweight='bold')
plt.xlabel('CGPA')
plt.ylabel('Count')
plt.legend()
plt.show()

In [7]:
# Plot 2: Department-wise CGPA Distribution (Boxplot with Stripplot)
plt.figure(figsize=(10, 5))
sns.boxplot(x='Department', y='CGPA', data=df_students, palette='Set2')
sns.stripplot(x='Department', y='CGPA', data=df_students, color='black', size=6, alpha=0.6, jitter=0.2)
plt.title('CGPA Spread Across Engineering Departments', fontsize=12, fontweight='bold')
plt.xticks(rotation=15)
plt.show()

In [8]:
# Plot 3: Study Hours vs CGPA (Scatter with Regression Fit)
plt.figure(figsize=(9, 4.5))
sns.regplot(x='StudyHoursPerWeek', y='CGPA', data=df_students, color='teal', scatter_kws={'s': 60})
plt.title('Weekly Study Hours vs. CGPA (Regression Analysis)', fontsize=12, fontweight='bold')
plt.xlabel('Study Hours per Week (hrs)')
plt.ylabel('CGPA')
plt.show()

In [9]:
# Plot 4: Correlation Heatmap of Quantitative Academic Metrics
numeric_features = ['Age', 'Year', 'CGPA', 'AttendanceRate', 'StudyHoursPerWeek', 'MathScore', 'ReadingScore', 'WritingScore']
plt.figure(figsize=(8, 6))
sns.heatmap(df_students[numeric_features].corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Correlation Matrix of Student Features', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.show()

---
## 6. Key Conclusions & Findings
1. **Study Hours vs CGPA**: Strong positive linear correlation exists between weekly independent study hours and final CGPA.
2. **Attendance Impact**: Consistent class attendance directly correlates with high exam performance in standardized quantitative and reading assessments.
3. **Department Diversity**: Data Science and Computer Science departments exhibit high average CGPAs, while mechanical and civil exhibit higher variance.
4. **Readiness for ML (Week 7–10)**: The cleaned, merged, and vectorized features prepared here are ideal for regression, classification, and clustering tasks in the subsequent weeks.